# Object Tracking using HSV Color Space
**Full Coding available at [Object_tracking.py](Object_tracking.py)** 
<br>
This is the documentation for Object Tracking using OpenCV with HSV color-based detection.

This project demonstrates real-time object tracking using color detection in the HSV color space. The system can track colored objects and provide pan-tilt camera control to follow the detected objects automatically.

***Libraries Used In The Script***
- OpenCV (cv2) - For image processing and computer vision
- Numpy (np) - For numerical operations and array handling
- Picamera2 - Raspberry Pi camera interface
- libcamera - Camera control library
- RPi_Robot_Hat_Lib - Custom robot control library

## Installing Libraries & Dependencies 
- Built-in Libraries:
    - Picamera2 (Pre-installed on Raspberry Pi OS)
    - libcamera (Pre-installed on Raspberry Pi OS)
    - Numpy (Usually pre-installed)
- Libraries Requiring Installation:
    - OpenCV
    - RPi_Robot_Hat_Lib (Custom library)

## Installing Required Libraries 
1. **OpenCV** [OpenCV.org](https://docs.opencv.org/4.x/d2/de6/tutorial_py_setup_in_ubuntu.html)
    - In the terminal type: `pip install opencv-python` 
    - To verify installation, in a Python file type: <br>
    `import cv2` <br>
    `print(cv2.__version__)`
    - This will print the version of OpenCV library installed  

2. **RPi_Robot_Hat_Lib**
    - This is a custom library for robot control
    - Install according to your robot hardware setup instructions

## Let's Start Coding !  
### 1. Import the Required Libraries  

The following libraries are required for the robot’s camera processing and motor control:  

- **OpenCV (`cv2`)** → Computer vision and image processing  
- **NumPy (`np`)** → Array and numerical operations  
- **Picamera2** → Raspberry Pi camera library  
- **libcamera** → Camera controls and transformations  
- **RPi_Robot_Hat_Lib** → Custom library for motor and encoder control  
- **time** → Timing and delays  

In [ ]:
import cv2
import time
from picamera2 import Picamera2
from libcamera import controls
from RPi_Robot_Hat_Lib import RobotController
import numpy as np

### 2. System Initialization  

This step initializes the **camera preview**, **motor controller**, and **pan-tilt servos** for the robot vision system.  

- **Global Variables**:  
  - `picam` → Picamera2 instance used for capturing frames  
  - `Motor` → Instance of `RobotController` for controlling servos and motors  
  - `vertical` → Servo number controlling vertical movement  
  - `horizontal` → Servo number controlling horizontal movement  

- **Usage**:  
  - Creates and configures the **Picamera2** object  
    ```python
    picam = Picamera2()
    config = picam.create_preview_configuration(
        main={"format": 'XRGB8888', "size": (640, 480)},
        transform=Transform(vflip=1)
    )
    picam.configure(config)
    picam.start()
    picam.set_controls({"AfMode": controls.AfModeEnum.Continuous})
    ```
  - Initializes the **Robot Controller** instance  
    ```python
    Motor = RobotController()
    ```
  - Sets the **pan and tilt servos** to neutral positions (90° each)  
    ```python
    vertical = 2
    horizontal = 1
    Motor.set_servo(vertical, 90)
    Motor.set_servo(horizontal, 90)
    ```


In [ ]:
picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={"format": 'XRGB8888', "size": (640, 480)}))
picam.start()
picam.set_controls({"AfMode": controls.AfModeEnum.Continuous})
Motor = RobotController()

vertical = 2
horizontal = 1
Motor.set_servo(vertical, 180)
Motor.set_servo(horizontal, 90)

### 3. The Function `colorPicker()`

This function creates an **interactive HSV color range picker** using OpenCV trackbars to dynamically adjust the lower and upper HSV thresholds for color detection.  

- **Function Name**: `colorPicker`  

- **Returns**:  
  - `tuple (numpy.ndarray, numpy.ndarray)` → A pair of arrays representing the **lower HSV bound** and **upper HSV bound**  

- **Global Variables**:  
  - `picam` → The `Picamera2` object used to capture live frames from the camera  

- **Usage**:  
  - Defines a placeholder callback for OpenCV trackbars  
    ```python
    def nothing(x):
        pass
    ```
  - Creates a window and HSV trackbars for **Lower Hue, Saturation, Value** and **Upper Hue, Saturation, Value**  
    ```python
    cv2.namedWindow("Color_Picker")
    cv2.createTrackbar("Lower Hue", "Color_Picker", 0, 179, nothing)
    cv2.createTrackbar("Lower Saturation", "Color_Picker", 0, 255, nothing)
    cv2.createTrackbar("Lower Value", "Color_Picker", 0, 255, nothing)
    cv2.createTrackbar("Upper Hue", "Color_Picker", 179, 179, nothing)
    cv2.createTrackbar("Upper Saturation", "Color_Picker", 255, 255, nothing)
    cv2.createTrackbar("Upper Value", "Color_Picker", 255, 255, nothing)
    ```
  - Captures frames from the camera, converts to **HSV**, and applies the mask dynamically based on the trackbar values  
    ```python
    img = picam.capture_array()
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    l_h = cv2.getTrackbarPos("Lower Hue", "Color_Picker")
    l_s = cv2.getTrackbarPos("Lower Saturation", "Color_Picker")
    l_v = cv2.getTrackbarPos("Lower Value", "Color_Picker")
    u_h = cv2.getTrackbarPos("Upper Hue", "Color_Picker")
    u_s = cv2.getTrackbarPos("Upper Saturation", "Color_Picker")
    u_v = cv2.getTrackbarPos("Upper Value", "Color_Picker")
    lower_bound = np.array([l_h, l_s, l_v])
    upper_bound = np.array([u_h, u_s, u_v])
    ```
  - Applies masking and displays the **stacked results** (mask, original, and result images side by side)  
    ```python
    mask = cv2.inRange(hsv, lower_bound, upper_bound) 
    res = cv2.bitwise_and(img, img, mask=mask)
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGRA)
    stacked = np.hstack((mask, img, res))
    cv2.imshow("Color_Picker", cv2.resize(stacked, None, fx=0.4, fy=0.4))
    ```
  - Exits when the user presses **'s'** and returns the selected HSV bounds  
    ```python
    return lower_bound, upper_bound
    ```


In [ ]:
def colorPicker(): 
    global picam 
    def nothing(x):
        pass    
    cv2.namedWindow("Color_Picker")
    cv2.createTrackbar("Lower Hue", "Color_Picker", 0, 179, nothing)
    cv2.createTrackbar("Lower Saturation", "Color_Picker", 0, 255, nothing)
    cv2.createTrackbar("Lower Value", "Color_Picker", 0, 255, nothing)
    cv2.createTrackbar("Upper Hue", "Color_Picker", 179, 179, nothing)
    cv2.createTrackbar("Upper Saturation", "Color_Picker", 255, 255, nothing)
    cv2.createTrackbar("Upper Value", "Color_Picker", 255, 255, nothing)
    while True:
        img = picam.capture_array()
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        l_h = cv2.getTrackbarPos("Lower Hue", "Color_Picker")
        l_s = cv2.getTrackbarPos("Lower Saturation", "Color_Picker")
        l_v = cv2.getTrackbarPos("Lower Value", "Color_Picker")
        u_h = cv2.getTrackbarPos("Upper Hue", "Color_Picker")
        u_s = cv2.getTrackbarPos("Upper Saturation", "Color_Picker")
        u_v = cv2.getTrackbarPos("Upper Value", "Color_Picker")
        lower_bound = np.array([l_h, l_s, l_v])
        upper_bound = np.array([u_h, u_s, u_v])
        mask = cv2.inRange(hsv, lower_bound, upper_bound) 
        res = cv2.bitwise_and(img, img, mask=mask)
        mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGRA)
        stacked = np.hstack((mask, img, res))
        cv2.imshow("Color_Picker", cv2.resize(stacked, None, fx=0.4, fy=0.4))
        if cv2.waitKey(1) & 0xFF == ord('s'):
            break
    return lower_bound, upper_bound 


### 4. The Function `main()`

This function runs the **core object detection and robot tracking loop**. It processes live camera frames, isolates objects based on HSV thresholds, calculates object position, and adjusts robot movement accordingly.

- **Function Name**: `main`

- **Global Variables**:  
  - `picam` → The **Picamera2 object** for real-time video capture  
  - `Motor` → The **RobotController object** for motor control  


- **Usage**:  

  1. **Get HSV Thresholds**  
     - Calls `colorPicker()` to interactively determine the **HSV lower and upper bounds**.  
       These values define which color range is treated as the target object.  
     ```python
     lower_bound , upper_bound = colorPicker()
     ```

  2. **Frame Capture & HSV Conversion**  
     - Continuously captures frames from the PiCamera.  
     - Converts frames from **BGR → HSV** color space because HSV is more robust for color detection under different lighting.  
     - Creates a **binary mask**: pixels within `[lower_bound, upper_bound]` are white (255), all others black (0).  
     ```python
     img = picam.capture_array()
     hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
     mask = cv2.inRange(hsv, lower_bound, upper_bound)
     ```

  3. **Contour Detection**  
     - Uses `cv2.findContours` to find **blobs of white pixels** in the mask.  
     - If contours are found, selects the **largest contour by area** as the main object.  
     - Ignores small contours (`area < 1500`) to reduce noise.  
     ```python
     contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
     largest_contour = max(contours, key=cv2.contourArea)
     area = cv2.contourArea(largest_contour)
     ```

  4. **Bounding Box & Centroid Calculation**  
     - Draws a rectangle around the detected object using `cv2.boundingRect`.  
     - Calculates the **centroid (center_x, center_y)**:  
       - `center_x = x + w/2`  
       - `center_y = y + h/2`  
     - Displays these values and area as debug text on the frame.  
     ```python
     x, y, w, h = cv2.boundingRect(largest_contour)
     center_x = int(x + w // 2)
     center_y = int(y + h // 2)
     ```

  5. **Decision Making: Robot Navigation**  
     - The **Y-axis (vertical position)** determines if the object is in the **upper safe region** (`center_y < 400`).  
     - The **X-axis (horizontal position)** is divided into 3 zones for navigation:  
       - **Left Zone** (`50 < center_x < 320`) → Turn Right  
       - **Center Zone** (`320 <= center_x <= 400`) → Move Forward  
       - **Right Zone** (`400 < center_x < 600`) → Turn Left  
       - Otherwise → Stop (object out of bounds)  
     - If the object drops below the safe zone (`center_y >= 400`) → Stop  
     ```python
     if center_y < 400:
         if 50 < center_x < 320:
             Motor.move(speed=0, turn=30)   # Turn right
         elif 400 < center_x < 600:
             Motor.move(speed=0, turn=-30)  # Turn left
         elif 320 <= center_x <= 400:
             Motor.Forward(20)              # Move straight
         else:
             Motor.Brake()                  # Out of range
     else:
         Motor.Brake()                      # Too low
     ```

  6. **Visual Feedback**  
     - Displays the **binary mask** and the **result image with bounding box** in two OpenCV windows for debugging.  
     - Press **`q`** to safely exit the loop.  
     ```python
     cv2.imshow("Camera", mask)
     cv2.imshow("Result", img)
     if cv2.waitKey(1) == ord('q'):
         break
     ```


In [ ]:
def main():
    global picam 
    lower_bound , upper_bound = colorPicker()

    while True:
        img = picam.capture_array()
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, lower_bound, upper_bound)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            for i, contour in enumerate(contours):
                area = cv2.contourArea(contour)
                if area > 1500:
                    x, y, w, h = cv2.boundingRect(contour)
                    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    cv2.putText(img, f"Object {i + 1}", (x, y - 10), cv2.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)

                    center_x = int(x + w // 2)
                    center_y = int(y + h // 2)
                    print("Center X:", center_x)
                    print("Center Y:", center_y)
                    print("Area:", area)

                    cv2.putText(img, f"Center X: {center_x}", (10, 30), cv2.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)
                    cv2.putText(img, f"Center Y: {center_y}", (10, 60), cv2.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)
                    cv2.putText(img, f"Area: {area}", (10, 90), cv2.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)

                    if center_y < 300:
                        if 50 < center_x < 320:
                            print("Turn right")
                            Motor.move(speed=0, turn=-20)
                        elif 400 < center_x < 600:
                            print("Turn left")
                            Motor.move(speed=0, turn=20)
                        elif 320 <= center_x <= 400:
                            Motor.Forward(20)
                            print("Centered")
                        else:
                            print("Out of range")
                            Motor.Brake() 
                    else:
                        print("Out of range")
                        Motor.Brake()

        cv2.imshow("Camera", mask)
        cv2.imshow("Result", img)
        if cv2.waitKey(1) == ord('q'):
            break

### 6. Program Execution Block

This block ensures the **main loop** runs when the script is executed directly and handles safe termination of hardware resources.

- **Function Name**: `__main__` execution block  

- **Parameters**:  
  - *None*  

- **Returns**:  
  - *None*  

- **Usage**:  
  - Executes the `main()` function only if the script is run directly  
    ```python
    if __name__ == '__main__':
        main()
    ```
  - Wraps execution in a `try/except` block to catch keyboard interruptions  
    ```python
    try:
        if __name__ == '__main__':
            main()
    except KeyboardInterrupt:
        print("KeyboardInterrupt")
    ```
  - Ensures all resources are properly released in the `finally` block:  
    - Stops the **camera** to free Picamera2 hardware  
      ```python
      picam.stop()
      ```
    - Cleans up the **motor controller and GPIO** to prevent pin lockup  
      ```python
      Motor.cleanup()
      ```
    - Destroys all **OpenCV windows** to free memory  
      ```python
      cv2.destroyAllWindows()
      ```
    - Prints termination status to the console for user feedback  
      ```python
      print("Program Terminated \n Exiting....")
      ```


In [ ]:
try:
    if __name__ == '__main__':
        main()
except KeyboardInterrupt:
    print("KeyboardInterrupt")
finally:
    picam.stop() 
    Motor.cleanup()
    cv2.destroyAllWindows()
    print("Program Terminated \n Exiting....")
